<a href="https://colab.research.google.com/github/djtheconqueror/home_ai/blob/main/SafeLine_V1_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SafeLine V1 — Streamlit Prototype

SafeLine V1 turns the original SafeLine V0 logic simulator into a clickable Streamlit prototype.

The goal is to demonstrate the product experience: generating a temporary SafeLine number, receiving simulated calls/texts during Live Mode, entering Review Mode after the user leaves, and allowing, muting, blocking, or burning contacts.

In [ ]:
!pip install streamlit pyngrok pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 64.3 MB/s eta 0:00:00


In [ ]:
%%writefile safeline_app.py

import streamlit as st
import random
import pandas as pd
from datetime import datetime, timedelta

st.set_page_config(
    page_title="SafeLine V1",
    page_icon="🛡️",
    layout="wide"
)

# -----------------------------
# Session State Setup
# -----------------------------

if "safeline_state" not in st.session_state:
    st.session_state.safeline_state = {
        "safe_number": None,
        "mode": "INACTIVE",
        "created_at": None,
        "expires_at": None,
        "contacts": {}
    }

state = st.session_state.safeline_state


# -----------------------------
# Helper Functions
# -----------------------------

def generate_safe_number():
    area_code = random.choice(["214", "305", "404", "617", "737", "786"])
    prefix = random.randint(200, 999)
    line = random.randint(1000, 9999)
    return f"({area_code}) {prefix}-{line}"


def start_live_session(duration_minutes=120):
    state["safe_number"] = generate_safe_number()
    state["mode"] = "LIVE"
    state["created_at"] = datetime.now()
    state["expires_at"] = datetime.now() + timedelta(minutes=duration_minutes)
    state["contacts"] = {}


def end_night():
    if state["mode"] not in ["INACTIVE", "BURNED"]:
        state["mode"] = "REVIEW"


def burn_safe_number():
    state["mode"] = "BURNED"


def get_or_create_contact(caller_name, caller_number):
    if caller_number not in state["contacts"]:
        state["contacts"][caller_number] = {
            "name": caller_name,
            "number": caller_number,
            "texts": 0,
            "calls": 0,
            "status": "ACTIVE",
            "history": []
        }

    return state["contacts"][caller_number]


def simulate_contact(caller_name, caller_number, contact_type="text", message=""):
    if state["mode"] == "INACTIVE":
        return "No active SafeLine session."

    if state["mode"] == "BURNED":
        return "SafeLine number has been burned. No contact can come through."

    contact = get_or_create_contact(caller_name, caller_number)

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if contact["status"] == "BURNED":
        action = "SESSION DEAD — contact rejected"
    elif contact["status"] == "BLOCKED":
        action = "BLOCKED — contact did not reach user"
    elif contact["status"] == "MUTED":
        action = "MUTED — contact logged silently"
    elif state["mode"] == "LIVE":
        action = "FORWARDED — user receives normal notification"
    elif state["mode"] == "REVIEW":
        action = "LOGGED — review mode active"
    elif state["mode"] == "SHIELD":
        action = "HELD — contact requires user review"
    else:
        action = "LOGGED"

    if contact_type == "text":
        contact["texts"] += 1
        event_type = "TEXT"
    else:
        contact["calls"] += 1
        event_type = "CALL"

    event = {
        "time": now,
        "type": event_type,
        "message": message,
        "action": action
    }

    contact["history"].append(event)

    return action


def update_contact_status(caller_number, new_status):
    if caller_number in state["contacts"]:
        state["contacts"][caller_number]["status"] = new_status


def contact_summary_df():
    rows = []

    for number, contact in state["contacts"].items():
        rows.append({
            "name": contact["name"],
            "number": contact["number"],
            "texts": contact["texts"],
            "calls": contact["calls"],
            "status": contact["status"],
            "total_attempts": contact["texts"] + contact["calls"],
        })

    if not rows:
        return pd.DataFrame(columns=["name", "number", "texts", "calls", "status", "total_attempts"])

    return pd.DataFrame(rows).sort_values("total_attempts", ascending=False)


# -----------------------------
# UI
# -----------------------------

st.title("SafeLine V1 — Temporary Safety Number Simulator")

st.markdown("""
SafeLine is a product prototype for safer temporary communication.
During **Live Mode**, calls and texts come through normally so the number behaves naturally in social situations.
After the user leaves, they can enter **Review Mode** and choose who to allow, mute, block, or burn.
""")

st.divider()

col1, col2, col3 = st.columns(3)

with col1:
    st.metric("SafeLine Number", state["safe_number"] if state["safe_number"] else "Not Started")

with col2:
    st.metric("Mode", state["mode"])

with col3:
    if state["expires_at"]:
        st.metric("Expires At", state["expires_at"].strftime("%H:%M:%S"))
    else:
        st.metric("Expires At", "N/A")

st.divider()

left, right = st.columns([1, 2])

# -----------------------------
# Controls
# -----------------------------

with left:
    st.subheader("Session Controls")

    duration = st.slider("Session length in minutes", 30, 240, 120, step=30)

    if st.button("Start Live Mode", use_container_width=True):
        start_live_session(duration)
        st.success("SafeLine session started.")

    if st.button("End Night / Review Mode", use_container_width=True):
        end_night()
        st.warning("SafeLine is now in Review Mode.")

    if st.button("Burn Whole SafeLine Number", use_container_width=True):
        burn_safe_number()
        st.error("SafeLine number burned.")

    st.divider()

    st.subheader("Simulate Incoming Contact")

    caller_name = st.text_input("Caller name", value="Nick")
    caller_number = st.text_input("Caller number", value="305-555-4832")
    contact_type = st.radio("Contact type", ["text", "call"], horizontal=True)
    message = st.text_input("Message", value="Hey this is me from the party")

    if st.button("Simulate Contact", use_container_width=True):
        action = simulate_contact(
            caller_name=caller_name,
            caller_number=caller_number,
            contact_type=contact_type,
            message=message
        )
        st.info(action)

# -----------------------------
# Dashboard
# -----------------------------

with right:
    st.subheader("Contact Dashboard")

    summary_df = contact_summary_df()

    if summary_df.empty:
        st.write("No contacts yet. Start Live Mode and simulate a contact.")
    else:
        st.dataframe(summary_df, use_container_width=True)

        st.subheader("Manage Contact")

        selected_number = st.selectbox(
            "Select contact number",
            summary_df["number"].tolist()
        )

        action_col1, action_col2, action_col3, action_col4 = st.columns(4)

        with action_col1:
            if st.button("Allow", use_container_width=True):
                update_contact_status(selected_number, "ACTIVE")
                st.success(f"{selected_number} allowed.")

        with action_col2:
            if st.button("Mute", use_container_width=True):
                update_contact_status(selected_number, "MUTED")
                st.warning(f"{selected_number} muted.")

        with action_col3:
            if st.button("Block", use_container_width=True):
                update_contact_status(selected_number, "BLOCKED")
                st.error(f"{selected_number} blocked.")

        with action_col4:
            if st.button("Burn Session", use_container_width=True):
                update_contact_status(selected_number, "BURNED")
                st.error(f"{selected_number} session burned.")

        st.subheader("Contact History")

        selected_contact = state["contacts"].get(selected_number)

        if selected_contact and selected_contact["history"]:
            history_df = pd.DataFrame(selected_contact["history"])
            st.dataframe(history_df, use_container_width=True)
        else:
            st.write("No history for this contact yet.")

st.divider()

st.subheader("Product Logic")

st.markdown("""
- **Live Mode:** contacts are forwarded normally
- **Review Mode:** contacts are logged for user decision
- **Muted:** contacts are logged silently
- **Blocked:** contacts do not reach the user
- **Burned Session:** one caller loses access
- **Burned Number:** the full SafeLine alias is disabled
""")

Writing safeline_app.py


In [ ]:
!streamlit run safeline_app.py --server.port 8501



2026-09-05 19:55:45.459 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.44.191:8501



In [ ]:
from pyngrok import ngrok
import os
import time

ngrok.kill()

public_url = ngrok.connect(8501)
print("SafeLine Streamlit App:", public_url)

os.system("streamlit run safeline_app.py --server.port 8501 &")
time.sleep(3)

# SafeLine V1 — Project Summary

SafeLine V1 is a clickable Streamlit prototype based on the original SafeLine V0 logic simulator.

The product concept is a temporary safety-number experience for safer first-contact communication.

During Live Mode, simulated calls and texts come through normally so the interaction does not feel suspicious or awkward in the moment. After the user leaves, they can enter Review Mode and decide who to allow, mute, block, or burn.

## Current Capabilities

- Generate a temporary SafeLine number
- Start Live Mode
- Simulate incoming calls and texts
- Track each contact separately
- Display a contact dashboard
- View contact history
- End the night and enter Review Mode
- Allow, mute, block, or burn individual contact sessions
- Burn the full SafeLine number

## Product Logic

- Live Mode: contacts are forwarded normally
- Review Mode: contacts are logged for user decision
- Muted: contact is logged silently
- Blocked: contact does not reach the user
- Burned Session: one caller loses access
- Burned Number: the full SafeLine alias is disabled

## Next Steps

- Add a database for persistent contacts and sessions
- Improve the UI design
- Add user settings
- Add privacy/security notes
- Explore Twilio integration for real masked-number testing
- Eventually build a mobile interface

This version does not use real phone numbers. It is a product-flow simulator for testing the core experience before telecom integration.